# Zomato India Restaurant Analysis: Data Cleaning
**Goal:** turn the raw Kaggle Zomato file into a clean, India-only dataset ready for analysis and Power BI.

**Input:** `data/raw/zomato.csv`, `data/raw/Country-Code.xlsx`  
**Output:** `data/processed/zomato_india_clean.csv`, `data/processed/restaurant_cuisines.csv`

In [1]:
import pandas as pd
print(pd.__version__)

2.3.3


In [2]:
df = pd.read_csv('../data/raw/zomato.csv', encoding='latin-1')

The file is not UTF-8 encoded, so I load it with `encoding='latin-1'`.

In [3]:
df.head()

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


In [4]:
df.shape

(9551, 21)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   object 
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   object 
 4   Address               9551 non-null   object 
 5   Locality              9551 non-null   object 
 6   Locality Verbose      9551 non-null   object 
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   object 
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   object 
 12  Has Table booking     9551 non-null   object 
 13  Has Online delivery   9551 non-null   object 
 14  Is delivering now     9551 non-null   object 
 15  Switch to order menu 

In [6]:
df['Country Code'].value_counts()

Country Code
1      8652
216     434
215      80
30       60
189      60
214      60
148      40
208      34
14       24
162      22
94       21
166      20
184      20
191      20
37        4
Name: count, dtype: int64

In [7]:
df[df['Rating text'] == 'Not rated']['Aggregate rating'].value_counts()

Aggregate rating
0.0    2148
Name: count, dtype: int64

**Findings from the first look**
- 9,551 rows and 21 columns; 8,652 restaurants (91%) are in India (Country Code 1).
- All 2,148 restaurants with `Rating text` = "Not rated" have `Aggregate rating` = 0. A 0 here means **no rating**, not a bad rating, so it must be fixed before calculating averages.

In [8]:
country = pd.read_excel('../data/raw/Country-Code.xlsx')
country

,Country Code,Country
0,1,India
1,14,Australia
2,30,Brazil
3,37,Canada
4,94,Indonesia
5,148,New Zealand
6,162,Phillipines
7,166,Qatar
8,184,Singapore
9,189,South Africa


In [9]:
df = df.merge(country, on='Country Code', how='left')
df = df[df['Country'] == 'India'].copy()
df.shape

(8652, 22)

In [10]:
cols = ['Restaurant ID', 'Restaurant Name', 'City', 'Locality', 'Cuisines',
        'Average Cost for two', 'Has Table booking', 'Has Online delivery',
        'Price range', 'Aggregate rating', 'Rating text', 'Votes',
        'Latitude', 'Longitude']
df = df[cols]
df.shape

(8652, 14)

In [11]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
df.columns

Index(['restaurant_id', 'restaurant_name', 'city', 'locality', 'cuisines',
       'average_cost_for_two', 'has_table_booking', 'has_online_delivery',
       'price_range', 'aggregate_rating', 'rating_text', 'votes', 'latitude',
       'longitude'],
      dtype='object')

In [12]:
print(df.duplicated().sum())
print(df['restaurant_id'].duplicated().sum())

0
0


In [13]:
df.isnull().sum()

restaurant_id           0
restaurant_name         0
city                    0
locality                0
cuisines                0
average_cost_for_two    0
has_table_booking       0
has_online_delivery     0
price_range             0
aggregate_rating        0
rating_text             0
votes                   0
latitude                0
longitude               0
dtype: int64

**Finding:** the full dataset had 9 missing `Cuisines` values, but all of them were outside India. After filtering, there are no missing values and no duplicate restaurants.

In [14]:
import numpy as np

df['is_rated'] = (df['aggregate_rating'] > 0).astype(int)
df.loc[df['aggregate_rating'] == 0, 'aggregate_rating'] = np.nan

print(df['is_rated'].value_counts())
print(df['aggregate_rating'].mean())

is_rated
1    6513
0    2139
Name: count, dtype: int64
3.3520343927529557


**Finding:** before the fix, the average rating was **2.52**. After treating 0 as "not rated", it is **3.35**, a difference of 0.83 stars. Keeping the zeros would have made every rating number in this project wrong.

I used `NaN` instead of dropping these rows, so unrated restaurants still count in "number of restaurants" but are skipped in averages.

In [15]:
print((df['average_cost_for_two'] == 0).sum())
df.nlargest(10, 'average_cost_for_two')[['restaurant_name', 'city', 'average_cost_for_two']]

9


,restaurant_name,city,average_cost_for_two
7543,Orient Express - Taj Palace Hotel,New Delhi,8000
4088,Tian - Asian Cuisine Studio - ITC Maurya,New Delhi,7000
4087,Bukhara - ITC Maurya,New Delhi,6500
7492,Nostalgia at 1911 Brasserie - The Imperial,New Delhi,6000
7494,1911 - The Imperial,New Delhi,6000
7499,The Spice Route - The Imperial,New Delhi,6000
7538,Wasabi by Morimoto - The Taj Mahal Hotel,New Delhi,6000
7522,MEGU - The Leela Palace,New Delhi,5500
7539,House of Ming - The Taj Mahal Hotel,New Delhi,5500
7512,24/7 Restaurant - The Lalit New Delhi,New Delhi,5100


In [16]:
for col in ['has_table_booking', 'has_online_delivery']:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

df[['has_table_booking', 'has_online_delivery']].value_counts()

has_table_booking  has_online_delivery
0                  0                      5545
                   1                      1996
1                  0                       684
                   1                       427
Name: count, dtype: int64

In [17]:
df['city'].value_counts()

city
New Delhi       5473
Gurgaon         1118
Noida           1080
Faridabad        251
Ghaziabad         25
Guwahati          21
Lucknow           21
Bhubaneshwar      21
Ahmedabad         21
Amritsar          21
Allahabad         20
Varanasi          20
Coimbatore        20
Chennai           20
Bhopal            20
Aurangabad        20
Bangalore         20
Agra              20
Goa               20
Kolkata           20
Jaipur            20
Kochi             20
Kanpur            20
Mangalore         20
Ludhiana          20
Indore            20
Dehradun          20
Vizag             20
Pune              20
Mumbai            20
Mysore            20
Nashik            20
Nagpur            20
Patna             20
Puducherry        20
Ranchi            20
Vadodara          20
Surat             20
Chandigarh        18
Hyderabad         18
Secunderabad       2
Mohali             1
Panchkula          1
Name: count, dtype: int64

## Fix restaurants with a cost of 0
A cost for two of ₹0 is not possible, so these 9 values are treated as missing (same approach as ratings).

In [18]:
df.loc[df['average_cost_for_two'] == 0, 'average_cost_for_two'] = np.nan
print('Zero costs left:', (df['average_cost_for_two'] == 0).sum())
print('Missing costs:', df['average_cost_for_two'].isnull().sum())

Zero costs left: 0
Missing costs: 9


**Decision on expensive restaurants:** the highest costs (₹5,000–₹8,000) all belong to 5-star hotel restaurants (Taj, ITC Maurya, The Imperial, The Leela). These are genuine prices, not errors, so I keep them. Because of these outliers, the **median** cost is a better "typical" value than the mean.

In [19]:
df['average_cost_for_two'].agg(['mean', 'median'])

mean      624.019438
median    450.000000
Name: average_cost_for_two, dtype: float64

## Merge twin cities
Secunderabad is Hyderabad's twin city, and Panchkula and Mohali are part of the Chandigarh tricity. Each has only 1–2 restaurants, so I merge them into the main city.

In [20]:
df['city'] = df['city'].replace({'Secunderabad': 'Hyderabad',
                                 'Panchkula': 'Chandigarh',
                                 'Mohali': 'Chandigarh'})
df['city'].value_counts().tail()

city
Ranchi      20
Surat       20
Vadodara    20
Varanasi    20
Vizag       20
Name: count, dtype: int64

**Note on coverage:** about 92% of restaurants are in Delhi NCR (New Delhi, Gurgaon, Noida, Faridabad, Ghaziabad). Most other cities have only ~20 restaurants each, so city comparisons outside Delhi NCR must be treated with caution.

In [21]:
ncr = ['New Delhi', 'Gurgaon', 'Noida', 'Faridabad', 'Ghaziabad']
df['region'] = np.where(df['city'].isin(ncr), 'Delhi NCR', 'Other cities')
df['region'].value_counts(normalize=True).round(3)

region
Delhi NCR       0.919
Other cities    0.081
Name: proportion, dtype: float64

## Create new columns
- `primary_cuisine`: the first cuisine listed for each restaurant
- `cuisine_count`: how many cuisines a restaurant serves
- `cost_bucket`: cost for two grouped into price bands

For rating groups I use the existing `rating_text` column (Excellent, Very Good, Good, Average, Poor, Not rated), since Zomato already provides it.

In [22]:
df['primary_cuisine'] = df['cuisines'].str.split(',').str[0].str.strip()
df['cuisine_count'] = df['cuisines'].str.split(',').str.len()

df['cost_bucket'] = pd.cut(df['average_cost_for_two'],
                           bins=[0, 300, 600, 1000, 2000, np.inf],
                           labels=['Up to 300', '301-600', '601-1000', '1001-2000', 'Above 2000'])

df[['primary_cuisine', 'cuisine_count', 'cost_bucket']].head()

,primary_cuisine,cuisine_count,cost_bucket
624,North Indian,2,601-1000
625,North Indian,2,601-1000
626,North Indian,1,301-600
627,Rajasthani,1,301-600
628,North Indian,3,601-1000


In [23]:
df['cost_bucket'].value_counts().sort_index()

cost_bucket
Up to 300     2716
301-600       3355
601-1000      1450
1001-2000      899
Above 2000     223
Name: count, dtype: int64

## Build a restaurant–cuisine table
One restaurant can serve several cuisines (for example "North Indian, Chinese, Mughlai"). To count cuisines correctly, I create a separate table with **one row per restaurant per cuisine**. In Power BI this becomes a bridge table.

In [24]:
cuisine_long = df[['restaurant_id', 'cuisines']].copy()
cuisine_long['cuisine'] = cuisine_long['cuisines'].str.split(',')
cuisine_long = cuisine_long.explode('cuisine')
cuisine_long['cuisine'] = cuisine_long['cuisine'].str.strip()
cuisine_long = cuisine_long[['restaurant_id', 'cuisine']]

print(cuisine_long.shape)
print('Unique cuisines:', cuisine_long['cuisine'].nunique())
cuisine_long['cuisine'].value_counts().head(10)

(17854, 2)
Unique cuisines: 90


cuisine
North Indian    3946
Chinese         2690
Fast Food       1963
Mughlai          992
Bakery           726
Continental      724
Italian          682
South Indian     631
Cafe             627
Desserts         597
Name: count, dtype: int64

## Validate and save

In [25]:
assert df['restaurant_id'].is_unique
assert df['aggregate_rating'].dropna().between(1, 5).all()
assert (df['average_cost_for_two'].dropna() > 0).all()
assert set(df['has_online_delivery'].unique()) <= {0, 1}
print('All checks passed')
df.isnull().sum()

All checks passed


restaurant_id              0
restaurant_name            0
city                       0
locality                   0
cuisines                   0
average_cost_for_two       9
has_table_booking          0
has_online_delivery        0
price_range                0
aggregate_rating        2139
rating_text                0
votes                      0
latitude                   0
longitude                  0
is_rated                   0
region                     0
primary_cuisine            0
cuisine_count              0
cost_bucket                9
dtype: int64

In [26]:
df = df.drop(columns='cuisines')
df.to_csv('../data/processed/zomato_india_clean.csv', index=False)
cuisine_long.to_csv('../data/processed/restaurant_cuisines.csv', index=False)
print('Saved:', df.shape, cuisine_long.shape)

Saved: (8652, 18) (17854, 2)


## Cleaning summary
| Step | Result |
|---|---|
| Raw data | 9,551 restaurants, 21 columns |
| Filtered to India | 8,652 restaurants |
| Duplicates | None found |
| Missing cuisines | None in India (9 were in other countries) |
| "Not rated" (rating 0) | 2,139 set to missing, flagged with `is_rated` |
| Cost of ₹0 | 9 set to missing |
| Expensive restaurants | Kept (genuine 5-star hotel prices) |
| Twin cities merged | Secunderabad → Hyderabad; Panchkula, Mohali → Chandigarh |
| New columns | `is_rated`, `region`, `primary_cuisine`, `cuisine_count`, `cost_bucket` |
| Final output | 8,652 restaurants + a restaurant–cuisine table |